# 03 — Weather-model training and evaluation: physics-loss ablation

One experiment per Kaggle session: choose `PHYSICS_LAMBDA`, `SEED` and `GRID`, run all cells.
Training uses the **unchanged internship code** (`src/climax/global_forecast/train.py`) with a run-specific copy of
`configs/regional_india.yaml`. Only data path, λ, seed, epochs, devices and output folders are changed.
The best checkpoint (by `val/w_rmse`) is then scored with `scripts/mtech/evaluate_forecast.py` against persistence and
train climatology on identical test pairs, including the hypsometric-consistency metric.

* `GRID = "corrected"`: the real 6×6 grid (lon 67.5–95.625°E). **Recommended for all new runs.** Attach the corrected dataset.
* `GRID = "orig_padded"`: reproduces the internship setup (6×5 + duplicated column). Attach the original dataset.

**`SMOKE_TEST = True` (default)** runs 20 training batches, 1 epoch and a 1-shard evaluation (~5 min) to check the pipeline.
Set it to `False` only for the real run (≈ 4–5 h on 2×T4). Use *Save Version → Save & Run All (Commit)* so the run does not depend on the browser.
Accelerator: **GPU T4 ×2**.

In [ ]:
# ---- Experiment ----
PHYSICS_LAMBDA = 0.001     # 0.0 = data-only baseline
SEED = 42
GRID = "corrected"         # "corrected" or "orig_padded"
MAX_EPOCHS = 20            # internship setting
DEVICES = 2                # 2 x T4 as in the internship; 1 also works (slower)
SMOKE_TEST = True          # set False for the real run

# ---- Optional: also score checkpoints trained earlier (attached as datasets) ----
# list of (checkpoint path, grid the model was trained on)
EXTRA_EVAL = [
    # ("/kaggle/input/.../lambda0/epoch_0XX.ckpt", "orig_padded"),
]

# ---- Paths (auto-discovered; override if ambiguous) ----
INPUT_ROOT = "/kaggle/input"
ORIG_DATA_OVERRIDE = None
CORRECTED_DATA_OVERRIDE = None
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
WORK = "/kaggle/working"

In [ ]:
import os, sys, json, subprocess, pathlib, importlib.util
REPO_DIR = f"{WORK}/ClimaX-Monsoon-PINN"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo commit:", COMMIT)

# Same pinned stack as the internship run (PL 1.8.6, timm 0.6.12, torchdata 0.8.0)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchdata", "timm", "pytorch-lightning", "jsonargparse"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdata==0.8.0", "timm==0.6.12",
                "pytorch-lightning==1.8.6", "jsonargparse[signatures]>=4.25.1"], check=True)
timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
txt = timm_init.read_text()
if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
    timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
sys.path.insert(0, f"{REPO_DIR}/src")
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
import numpy as np
from climax_hazards import inspect_utils as iu

cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT) if "fake_era5_demo" not in str(p)]
def pick(width, override, what):
    if override:
        return pathlib.Path(override)
    found = [p for p in cands if np.load(p / "lon.npy").size == width and not iu.detect_lon_padding(p)["looks_padded"]]
    return iu.resolve_single(found, None, what) if found else None
ORIG_DIR = pick(5, ORIG_DATA_OVERRIDE, "original 6x5 dataset")
CORR_DIR = pick(6, CORRECTED_DATA_OVERRIDE, "corrected 6x6 dataset")
print("original 6x5:", ORIG_DIR, "\ncorrected 6x6:", CORR_DIR)

def data_dirs(grid):
    # (training dir, unpadded evaluation dir)
    if grid == "corrected":
        assert CORR_DIR is not None, "attach the corrected 6x6 dataset"
        return CORR_DIR, CORR_DIR
    assert ORIG_DIR is not None, "attach climax-indian-monsoon-regional"
    padded = pathlib.Path(WORK) / "regional_climax_dataset_6x6"
    if not padded.exists():
        iu.make_padded_copy(ORIG_DIR, padded)  # same operation as the internship notebook, never applied twice
    return padded, ORIG_DIR

## Run configuration

In [ ]:
import yaml, copy
RUN = f"{GRID}_lambda{PHYSICS_LAMBDA:g}_seed{SEED}" + ("_smoke" if SMOKE_TEST else "")
RUN_DIR = f"{WORK}/runs/{RUN}"
os.makedirs(RUN_DIR, exist_ok=True)
train_dir, eval_dir = data_dirs(GRID)

cfg = yaml.safe_load(open(f"{REPO_DIR}/configs/regional_india.yaml"))
cfg["seed_everything"] = SEED
cfg["model"]["physics_lambda"] = float(PHYSICS_LAMBDA)
cfg["data"]["root_dir"] = str(train_dir)
t = cfg["trainer"]
t["max_epochs"] = MAX_EPOCHS
n_gpu = torch.cuda.device_count()
DEVICES = min(DEVICES, n_gpu) if n_gpu else 1
t["accelerator"] = "gpu" if n_gpu else "cpu"   # cpu only for local pipeline checks
t["devices"] = DEVICES
t["strategy"] = "ddp" if DEVICES > 1 else None
t["default_root_dir"] = RUN_DIR
t["logger"]["init_args"]["save_dir"] = f"{RUN_DIR}/logs"
for cb in t["callbacks"]:
    if cb["class_path"].endswith("ModelCheckpoint"):
        cb["init_args"]["dirpath"] = f"{RUN_DIR}/checkpoints"
if SMOKE_TEST:
    t.update({"max_epochs": 1, "limit_train_batches": 20, "limit_val_batches": 5, "limit_test_batches": 5})
CFG_PATH = f"{RUN_DIR}/config.yaml"
yaml.safe_dump(cfg, open(CFG_PATH, "w"), sort_keys=False)
json.dump({"run": RUN, "grid": GRID, "physics_lambda": PHYSICS_LAMBDA, "seed": SEED, "max_epochs": t["max_epochs"],
           "devices": DEVICES, "smoke_test": SMOKE_TEST, "repo_commit": COMMIT, "train_dir": str(train_dir),
           "eval_dir": str(eval_dir)}, open(f"{RUN_DIR}/run_info.json", "w"), indent=1)
print(open(f"{RUN_DIR}/run_info.json").read())

## Train (internship training script, unchanged)

In [ ]:
import re, time
log_path = f"{RUN_DIR}/train.log"
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src")
keep = re.compile(r"physics_lambda|global step|reached|not in top|Restoring|Loaded model|Error|Traceback|test/|stopped|max_epochs")
t0 = time.time()
with open(log_path, "w") as log:
    proc = subprocess.Popen([sys.executable, "src/climax/global_forecast/train.py", "--config", CFG_PATH],
                            cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        for part in line.replace("\r", "\n").split("\n"):
            if keep.search(part):
                print(part.strip()[-300:], flush=True)
    rc = proc.wait()
print(f"exit code {rc}, {(time.time() - t0) / 3600:.2f} h; full log: {log_path}")
assert rc == 0, "training failed - see the log above"

## Evaluate the best checkpoint

In [ ]:
import pandas as pd
ck_dir = pathlib.Path(RUN_DIR) / "checkpoints"
last = ck_dir / "last.ckpt"
info = iu.summarize_checkpoint(last)
mc = next(v for k, v in info["callbacks"].items() if k.startswith("ModelCheckpoint"))
BEST = pathlib.Path(mc["best_model_path"])
print("best:", BEST.name, "val/w_rmse =", mc["best_model_score"], "| last epoch:", info["epoch"])
json.dump({"best_checkpoint": BEST.name, "best_val_w_rmse": mc["best_model_score"], "last_epoch": info["epoch"],
           "best_k_models": mc["best_k_models"]}, open(f"{RUN_DIR}/selection.json", "w"), indent=1)

def run_eval(ckpt, grid, out_csv, config=None):
    _, edir = data_dirs(grid)
    cmd = [sys.executable, f"{REPO_DIR}/scripts/mtech/evaluate_forecast.py", "--data_dir", str(edir),
           "--config", config or f"{REPO_DIR}/configs/regional_india.yaml", "--checkpoint", str(ckpt), "--out", out_csv]
    if SMOKE_TEST:
        cmd += ["--max_files", "1"]
    r = subprocess.run(cmd, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    print(r.stdout[-6000:], r.stderr[-3000:])
    assert r.returncode == 0
    return pd.read_csv(out_csv)

ev = run_eval(BEST, GRID, f"{RUN_DIR}/eval_test.csv", CFG_PATH)

In [ ]:
def summary(ev, model_name):
    real = ev[ev["columns"] == "real"]
    piv = real.pivot_table(index="variable", columns="source", values="rmse", sort=False)
    out = pd.DataFrame({"model": piv[model_name], "persistence": piv["persistence"], "climatology": piv["climatology"]})
    out["skill_vs_clim_%"] = 100 * (1 - out["model"] / out["climatology"])
    return out

s = summary(ev, BEST.name)
print(f"{RUN}: test RMSE on real columns (hyps_resid_* = hypsometric residual error, metres)")
print(s.round(3).to_string())
s.to_csv(f"{RUN_DIR}/eval_summary.csv")

## Optional: score earlier checkpoints (e.g. your λ = 0 run)

In [ ]:
for ck, grid in EXTRA_EVAL:
    out = f"{WORK}/runs/eval_{pathlib.Path(ck).parent.name}_{pathlib.Path(ck).stem}.csv"
    e = run_eval(ck, grid, out)
    print(pathlib.Path(ck).name, "->", out)
    print(summary(e, pathlib.Path(ck).name).round(3).to_string())

In [ ]:
# Keep only what is needed: best checkpoint + logs/results (other checkpoints are ~1.3 GB each)
if not SMOKE_TEST:
    for p in ck_dir.glob("*.ckpt"):
        if p.name not in (BEST.name, "last.ckpt"):
            p.unlink()
print(subprocess.run(["du", "-sh", RUN_DIR], capture_output=True, text=True).stdout)
print(sorted(os.listdir(RUN_DIR)))